<a href="https://colab.research.google.com/github/Bhavika-G-Patil/learning-genai/blob/main/learning_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Hyperparameters
T = 4          # sequence length (number of tokens)
C = 8          # embedding size (size of each token vector)
head_size = 4  # size of Q, K, V vectors

In [ ]:
# Fake input: 4 tokens, each with embedding size 8
torch.manual_seed(42)  # so we all get same random numbers
x = torch.randn(T, C)  # shape: (T, C) = (4, 8)

print("Input shape:", x.shape)
print("Input tensor:\n", x)

Input shape: torch.Size([4, 8])
Input tensor:
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,  1.6806],
        [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599]])


In [ ]:
# Create Q, K, V projection matrices (learned weight matrices)
query = nn.Linear(C, head_size, bias=False)  # projects C→head_size
key   = nn.Linear(C, head_size, bias=False)  # projects C→head_size
value = nn.Linear(C, head_size, bias=False)  # projects C→head_size

# Pass our input through each to get Q, K, V vectors
q = query(x)  # shape: (T, head_size) = (4, 4)
k = key(x)    # shape: (T, head_size) = (4, 4)
v = value(x)  # shape: (T, head_size) = (4, 4)

print("Q shape:", q.shape)
print("K shape:", k.shape)
print("V shape:", v.shape)

Q shape: torch.Size([4, 4])
K shape: torch.Size([4, 4])
V shape: torch.Size([4, 4])


In [ ]:
# Compute raw attention scores — how much should each token attend to each other?
scores = q @ k.transpose(-2, -1)  # shape: (T, T) = (4, 4)

print("Attention scores shape:", scores.shape)
print("Raw attention scores:\n", scores)

Attention scores shape: torch.Size([4, 4])
Raw attention scores:
 tensor([[ 2.7821e-02,  7.2502e-01, -9.0515e-01,  3.7418e-01],
        [-2.1680e-01,  1.0294e-03, -1.6473e+00,  2.7978e-01],
        [ 9.0578e-01, -7.9611e-01, -3.5133e-01, -1.1463e+00],
        [ 7.7751e-02,  1.8723e-01,  1.3176e+00,  2.0374e-01]],
       grad_fn=<MmBackward0>)


In [ ]:
# Scale scores by square root of head_size
import math

scaled_scores = scores / math.sqrt(head_size)

print("Scaled scores shape:", scaled_scores.shape)
print("Scaled scores:\n", scaled_scores)

Scaled scores shape: torch.Size([4, 4])
Scaled scores:
 tensor([[ 1.3910e-02,  3.6251e-01, -4.5258e-01,  1.8709e-01],
        [-1.0840e-01,  5.1469e-04, -8.2364e-01,  1.3989e-01],
        [ 4.5289e-01, -3.9805e-01, -1.7567e-01, -5.7313e-01],
        [ 3.8875e-02,  9.3613e-02,  6.5878e-01,  1.0187e-01]],
       grad_fn=<DivBackward0>)


In [ ]:
# Create causal mask — block future tokens
tril = torch.tril(torch.ones(T, T))  # lower triangular matrix
print("Mask:\n", tril)

# Replace upper triangle with -infinity
masked_scores = scaled_scores.masked_fill(tril == 0, float('-inf'))
print("\nMasked scores:\n", masked_scores)

Mask:
 tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

Masked scores:
 tensor([[ 1.3910e-02,        -inf,        -inf,        -inf],
        [-1.0840e-01,  5.1469e-04,        -inf,        -inf],
        [ 4.5289e-01, -3.9805e-01, -1.7567e-01,        -inf],
        [ 3.8875e-02,  9.3613e-02,  6.5878e-01,  1.0187e-01]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
# Apply softmax to get attention weights
weights = F.softmax(masked_scores, dim=-1)

print("Attention weights:\n", weights)
print("\nRow sums (should all be 1.0):", weights.sum(dim=-1))

Attention weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4728, 0.5272, 0.0000, 0.0000],
        [0.5101, 0.2178, 0.2721, 0.0000],
        [0.2008, 0.2121, 0.3732, 0.2139]], grad_fn=<SoftmaxBackward0>)

Row sums (should all be 1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [ ]:
# Weighted sum of Value vectors
output = weights @ v  # shape: (T, head_size) = (4, 4)

print("Output shape:", output.shape)
print("Output (new context-aware embeddings):\n", output)

Output shape: torch.Size([4, 4])
Output (new context-aware embeddings):
 tensor([[ 1.2212, -0.2578,  0.5259,  0.8316],
        [ 0.5918, -0.0603,  0.1012,  0.5516],
        [ 0.6949,  0.0308,  0.0550,  0.3061],
        [ 0.2492,  0.0079, -0.3244, -0.1636]], grad_fn=<MmBackward0>)
